In [ ]:
"""
Improved SFT Training Script for Tunix Competition
- Proper checkpointing with resume capability
- W&B integration for monitoring
- Memory-efficient data handling
- Competition-compatible model saving
"""

!pip install -q kagglehub tensorflow datasets tiktoken wandb
!pip install "google-tunix[prod]==0.1.3"
!pip uninstall -q -y flax
!pip install -U flax

import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.1/75.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 620.7/620.7 MB 733.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.9/22.9 MB 93.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.0/201.0 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.2/208.2 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.3/150.3 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.1/411.1 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx
import optax
import numpy as np
import pandas as pd
import re
import time
import os
from datasets import load_dataset, Dataset
import grain.python as grain
import orbax.checkpoint as ocp
import wandb

from tunix.models.gemma3 import params, model as gemma_model
from tunix.sft.peft_trainer import PeftTrainer, TrainingConfig
from tunix.generate import sampler as sampler_lib
from tunix import MetricsLoggerOptions

print(f"JAX devices: {jax.devices()}")
print(f"Device count: {len(jax.devices())}")
print(f"Device type: {jax.devices()[0].platform}")

# JAX optimizations
os.environ['XLA_FLAGS'] = (
    '--xla_gpu_enable_triton_softmax_fusion=true '
    '--xla_gpu_triton_gemm_any=True '
    '--xla_gpu_enable_async_collectives=true'
)
os.environ['JAX_COMPILATION_CACHE_DIR'] = '/tmp/jax_cache'
jax.config.update('jax_enable_x64', False)

In [ ]:
# Training hyperparameters
MAX_SEQ_LENGTH = 1800
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-5
NUM_EPOCHS = 2
WARMUP_STEPS = 100
# N_SAMPLES = 20000

# Optimizer settings
ADAM_BETA1 = 0.9
ADAM_BETA2 = 0.999
ADAM_EPSILON = 1e-8
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0

# Google Drive paths (mount Drive first!)
DRIVE_ROOT = "/content/drive/MyDrive/kaggle-tunix-folder"
CHECKPOINT_DIR = f"{DRIVE_ROOT}/checkpoints"
FINAL_MODEL_DIR = "/content/drive/MyDrive/kaggle-tunix-folder/final_model_v5"


# Checkpointing and logging
SAVE_INTERVAL_STEPS = 500
EVAL_INTERVAL_STEPS = 100
MAX_CHECKPOINTS_TO_KEEP = 3

# W&B config
WANDB_PROJECT = "tunix-sft-training-colab-v2"
WANDB_RUN_NAME = f"gemma3-1b-distill-{int(time.time())}"

print(f"Configuration:")
print(f"  Max sequence length: {MAX_SEQ_LENGTH}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Gradient accumulation: {GRADIENT_ACCUMULATION_STEPS}")
print(f"  Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Save interval: {SAVE_INTERVAL_STEPS} steps")

In [ ]:
from google.colab import drive

# try:
#     drive.flush_and_unmount()
# except:
#     pass

drive.mount('/content/drive')


# # Mount Drive if not already mounted
# if not os.path.exists('/content/drive/MyDrive'):
#     drive.mount('/content/drive')
#     print("✓ Google Drive mounted successfully!")
# else:
#     print("✓ Google Drive already mounted")
# Create directories now that Drive is mounted
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

print(f"\nCheckpoint directory: {CHECKPOINT_DIR}")
print(f"Final model directory: {FINAL_MODEL_DIR}")

# Set dataset path directly
DATASET_PATH = f"{DRIVE_ROOT}/dataset/custom_df_train_sft.parquet"

print(f"\n{'='*60}")
print("LOADING DATASET")
print(f"{'='*60}")
print(f"Path: {DATASET_PATH}")

# Verify file exists
if not os.path.exists(DATASET_PATH):
    print(f"\n❌ File not found: {DATASET_PATH}")
    print("\nAvailable files in dataset folder:")
    dataset_folder = f"{DRIVE_ROOT}/dataset"
    for file in os.listdir(dataset_folder):
        print(f"  - {file}")
    raise FileNotFoundError(f"Dataset file not found: {DATASET_PATH}")

# Show file info
file_size_mb = os.path.getsize(DATASET_PATH) / (1024*1024)
print(f"File size: {file_size_mb:.1f} MB")
print(f"✓ File exists")
print(f"{'='*60}\n")

# Load dataset - parquet file with 'input' and 'output' columns
df_full = pd.read_parquet(DATASET_PATH)

# df_train_sample =df_full.sample(N_SAMPLES)
# df_train_sample =df_full

##Subset the training data by sampling records across domain
domain_counts = {
    'code': 2000,# Reduced from 6K - 2K
    'math': 2500,# Reduced from 10K - 2500 (low eval weight)
    'commonsense_reasoning': 6356, #max samples possible
    'creative_ideation': 1893,#max samples possible
    'creative_writing': 3637,#max samples possible
    'financial_reasoning': 1682,#max samples possible
    'numerical_reasoning': 1898,# Keep modest (some math-adjacent)
    'reading_comprehension': 947,#max samples possible
    'science': 4161,#max samples possible
    'summarization': 1715#max samples possible
}

# Sample each domain separately and concatenate
df_train_sample = []
for domain, count in domain_counts.items():
    domain_samples = df_full[df_full['domain'] == domain].sample(
        n=min(count, len(df_full[df_full['domain'] == domain])),
        random_state=42  # for reproducibility
    )
    df_train_sample.append(domain_samples)

# Concatenate all domain samples into a single DataFrame
df_train_sample = pd.concat(df_train_sample, ignore_index=True)

# shuffle after concatenating to mix domains
# This ensures the training doesn't see all samples from one domain in sequence, which can help with learning stability
df_train_sample = df_train_sample.sample(frac=1, random_state=29).reset_index(drop=True)

print(f"Loaded {len(df_train_sample)} total examples")
print(f"Columns: {df_train_sample.columns.tolist()}")



##Formar the reasoning and response into an output column
def format_output(reasoning, response):
    reasoning = str(reasoning) if not pd.isna(reasoning) else ""
    response = str(response) if not pd.isna(response) else ""
    return f"<reasoning>{reasoning.strip()}</reasoning><answer>{response.strip()}</answer>"


##Use cleaned_reasoning and respoinse
df_train_sample['output'] = df_train_sample.apply(
    lambda row: format_output(row['cleaned_reasoning'], row['response']),
    axis=1
)

# Verify required columns exist
required_columns = ['input', 'output']
missing_columns = [col for col in required_columns if col not in df_train_sample.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}. Found columns: {df_train_sample.columns.tolist()}")

print("\nSample row:")
print(f"Input: {df_train_sample['input'].iloc[0][:100]}...")
print(f"Output: {df_train_sample['output'].iloc[0]}...")


# No token counting needed - we'll filter during formatting if needed
print(f"\n✓ Dataset ready with {len(df_train_sample)} examples")

print(f"Total combined dataset size: {len(df_train_sample)}")
print("Dataset sizes by domain:")
for domain in df_train_sample['domain'].unique():
    domain_size = len(df_train_sample[df_train_sample['domain'] == domain])
    print(f"  {domain}: {domain_size}")

Mounted at /content/drive

Checkpoint directory: /content/drive/MyDrive/kaggle-tunix-folder/checkpoints
Final model directory: /content/drive/MyDrive/kaggle-tunix-folder/final_model_v5

LOADING DATASET
Path: /content/drive/MyDrive/kaggle-tunix-folder/dataset/custom_df_train_sft.parquet
File size: 84.3 MB
✓ File exists

Loaded 26789 total examples
Columns: ['input', 'source_answer', 'split', 'source', 'domain', 'ground_truth', 'problem_type', 'question_type', 'uid', 'reasoning', 'response', 'input_tokens', 'output_tokens', 'total_tokens', 'success', 'error', 'model_answer', 'model_answer_available', 'validated_correct', 'difficulty', 'input_token_count', 'extracted_answer', 'is_correct', 'story_token_count', 'category', 'cleaned_reasoning', 'low_reasoning_quality']

Sample row:
Input: Context:
Jingle All the Way: Jingle All the Way is a 1996 American Christmas family comedy film dire...
Output: <reasoning>This is a commonsense reasoning task.

We need to answer: Sinbad starred in what 1

In [ ]:
# ============================================================================
# Load Model and Tokenizer
# ============================================================================
MODEL_CP_PATH = params.GEMMA3_1B_IT
model_config = gemma_model.ModelConfig.gemma3_1b()

print("Loading Gemma 3 1B model...")
student_model = params.create_model_from_checkpoint(MODEL_CP_PATH, model_config)
gemma_tokenizer = params.create_tokenizer()

print(f"✓ Model loaded: {model_config.num_layers} layers")
print(f"✓ Tokenizer vocab size: {gemma_tokenizer.vocab_size}")

Loading Gemma 3 1B model...


✓ Model loaded: 26 layers
✓ Tokenizer vocab size: <bound method SentencePieceProcessor.vocab_size of <sentencepiece.SentencePieceProcessor; proxy of <Swig Object of type 'sentencepiece::SentencePieceProcessor *' at 0x7eff394a28e0> >>


In [ ]:
# ============================================================================
# Format and Tokenize Data
# ============================================================================
def format_for_gemma3(example):
    """Format into Gemma 3 chat template - matches V3 approach"""
    text = f"<start_of_turn>user\n{example['input']}<end_of_turn>\n<start_of_turn>model\n{example['output']}<end_of_turn>"
    return {'text': text}

# Add token length to each example using SentencePiece tokenizer
def add_token_length(batch):
    lengths = []
    for text in batch['text']:
        # SentencePieceProcessor uses .encode() method
        token_ids = gemma_tokenizer.encode(text)
        lengths.append(len(token_ids))
    return {'token_length': lengths}

# Convert to HuggingFace Dataset
hf_dataset = Dataset.from_pandas(df_train_sample)
hf_dataset = hf_dataset.map(format_for_gemma3)

print(f"Dataset size before filtering: {len(hf_dataset)}")
print("Computing token lengths...")
hf_dataset = hf_dataset.map(add_token_length, batched=True, batch_size=500)

# Show overall length distribution
lengths = hf_dataset['token_length']
print(f"\nOverall token length statistics (before filtering):")
print(f"  Min: {min(lengths)}")
print(f"  Max: {max(lengths)}")
print(f"  Mean: {sum(lengths)/len(lengths):.0f}")
print(f"  Median: {sorted(lengths)[len(lengths)//2]}")
print(f"  75th percentile: {sorted(lengths)[int(len(lengths)*0.75)]}")
print(f"  90th percentile: {sorted(lengths)[int(len(lengths)*0.90)]}")
print(f"  95th percentile: {sorted(lengths)[int(len(lengths)*0.95)]}")
print(f"  99th percentile: {sorted(lengths)[int(len(lengths)*0.99)]}")
print(f"  Samples > {MAX_SEQ_LENGTH}: {sum(1 for l in lengths if l > MAX_SEQ_LENGTH)}")

# Per-domain statistics
print(f"\nToken length distribution by domain:")
print(f"{'Domain':<25} {'Count':>6} {'Mean':>6} {'Median':>6} {'75th%':>6} {'90th%':>6} {'95th%':>6} {'>{MAX_SEQ_LENGTH}':>6}")
print("-" * 85)

for domain in sorted(hf_dataset.unique('domain')):
    domain_data = hf_dataset.filter(lambda x: x['domain'] == domain)
    domain_lengths = domain_data['token_length']

    if len(domain_lengths) > 0:
        sorted_lens = sorted(domain_lengths)
        count = len(domain_lengths)
        mean_len = sum(domain_lengths) / count
        median_len = sorted_lens[count // 2]
        p75 = sorted_lens[int(count * 0.75)]
        p90 = sorted_lens[int(count * 0.90)]
        p95 = sorted_lens[int(count * 0.95)]
        over_max = sum(1 for l in domain_lengths if l > MAX_SEQ_LENGTH)

        print(f"{domain:<25} {count:>6} {mean_len:>6.0f} {median_len:>6} {p75:>6} {p90:>6} {p95:>6} {over_max:>6}")


# Filter out samples exceeding MAX_SEQ_LENGTH
hf_dataset = hf_dataset.filter(lambda x: x['token_length'] <= MAX_SEQ_LENGTH)

print(f"\nDataset size after filtering: {len(hf_dataset)}")
print(f"Removed {len(df_train_sample) - len(hf_dataset)} samples")


# Split train/eval
split_dataset = hf_dataset.train_test_split(test_size=0.05, seed=42)
train_data = split_dataset['train']
eval_data = split_dataset['test']

print(f"Train: {len(train_data)}, Eval: {len(eval_data)}")
print(f"\nFormatted sample (first 300 chars):")
print(train_data[0]['text'])

# Clear memory
del df_train_sample, hf_dataset, split_dataset
import gc
gc.collect()

Map:   0%|          | 0/26789 [00:00<?, ? examples/s]

Dataset size before filtering: 26789
Computing token lengths...


Map:   0%|          | 0/26789 [00:00<?, ? examples/s]


Overall token length statistics (before filtering):
  Min: 87
  Max: 3850
  Mean: 798
  Median: 773
  75th percentile: 1146
  90th percentile: 1480
  95th percentile: 1655
  99th percentile: 1847
  Samples > 1800: 452

Token length distribution by domain:
Domain                     Count   Mean Median  75th%  90th%  95th% >{MAX_SEQ_LENGTH}
-------------------------------------------------------------------------------------


Filter:   0%|          | 0/26789 [00:00<?, ? examples/s]

...

Filter:   0%|          | 0/26789 [00:00<?, ? examples/s]

summarization               1715   1364   1385   1598   1728   1775     52


Filter:   0%|          | 0/26789 [00:00<?, ? examples/s]


Dataset size after filtering: 26337
Removed 452 samples
Train: 25020, Eval: 1317

Formatted sample (first 300 chars):
<start_of_turn>user
Question: Which are produced during photosynthesis?

Options:
A. carbon dioxide and minerals
B. carbon dioxide and sugar
C. oxygen and minerals
D. oxygen and sugar<end_of_turn>
<start_of_turn>model
<reasoning>This is a scientific reasoning task.

We need to answer which are produced during photosynthesis. Photosynthesis consumes CO2 and water, produces O2 and sugars (glucose). So answer D: oxygen and sugar. Provide explanation.</reasoning><answer>Photosynthesis is the process by which green plants, algae, and some bacteria convert carbon dioxide (CO₂) and water (H₂O) into organic compounds (mainly sugars such as glucose) using light energy. The overall simplified equation is:

\[
6\,\text{CO}_2 + 6\,\text{H}_2\text{O} \xrightarrow{\text{light}} \text{C}_6\text{H}_{12}\text{O}_6 + 6\,\text{O}_2
\]

- **Inputs (reactants):** carbon dioxide and water.


251

In [ ]:

def tokenize_batch(examples):
    """Tokenize with padding/truncation - exact V3 approach"""
    tokenized_ids = []
    attention_masks = []
    pad_id = gemma_tokenizer.pad_id()

    for text in examples['text']:
        tokens = gemma_tokenizer.encode(text)

        # Truncate if too long
        if len(tokens) > MAX_SEQ_LENGTH:
            tokens = tokens[:MAX_SEQ_LENGTH]

        # Create attention mask
        attention_mask = [1] * len(tokens)

        # Pad to max length
        padding_length = MAX_SEQ_LENGTH - len(tokens)
        tokens = tokens + [pad_id] * padding_length
        attention_mask = attention_mask + [0] * padding_length

        tokenized_ids.append(tokens)
        attention_masks.append(attention_mask)

    return {
        'input_ids': tokenized_ids,
        'attention_mask': attention_masks,
    }

print("Tokenizing...")
train_tokenized = train_data.map(tokenize_batch, batched=True, remove_columns=train_data.column_names, batch_size=50)
eval_tokenized = eval_data.map(tokenize_batch, batched=True, remove_columns=eval_data.column_names, batch_size=50)

print(f"✓ Tokenization complete")
print(f"  Train: {len(train_tokenized)} samples")
print(f"  Eval: {len(eval_tokenized)} samples")

# Clear more memory
del train_data, eval_data
gc.collect()


Tokenizing...


Map:   0%|          | 0/25020 [00:00<?, ? examples/s]

Map:   0%|          | 0/1317 [00:00<?, ? examples/s]

✓ Tokenization complete
  Train: 25020 samples
  Eval: 1317 samples


179

In [ ]:
# ============================================================================
# Create Tunix Format Datasets (Minimal Memory)
# ============================================================================
def create_tunix_dataset(tokenized_data):
    """Create dataset with minimal memory - trainer generates attention_mask"""
    examples = []

    for i in range(len(tokenized_data)):
        input_ids = tokenized_data[i]['input_ids']
        attention_mask = tokenized_data[i]['attention_mask']

        # Position indices
        positions = list(range(len(input_ids)))
        input_mask = attention_mask

        examples.append({
            'input_tokens': np.array(input_ids, dtype=np.int32),
            'input_mask': np.array(input_mask, dtype=np.float32),
            'positions': np.array(positions, dtype=np.int32),
            # REMOVED: attention_mask - trainer will generate causal mask
        })

    return examples

print("Creating Tunix format datasets (minimal memory)...")
train_examples = create_tunix_dataset(train_tokenized)
eval_examples = create_tunix_dataset(eval_tokenized)

print(f"✓ Datasets created")
print(f"  Keys: {list(train_examples[0].keys())}")
print(f"  One example size: {sum(v.nbytes for v in train_examples[0].values()) / 1e6:.2f}MB")

# Memory check
import sys
total_train_mb = sum(sum(v.nbytes for v in ex.values()) for ex in train_examples) / 1e6
total_eval_mb = sum(sum(v.nbytes for v in ex.values()) for ex in eval_examples) / 1e6
print(f"  Total train memory: {total_train_mb:.0f}MB")
print(f"  Total eval memory: {total_eval_mb:.0f}MB")



Creating Tunix format datasets (minimal memory)...
✓ Datasets created
  Keys: ['input_tokens', 'input_mask', 'positions']
  One example size: 0.02MB
  Total train memory: 540MB
  Total eval memory: 28MB


In [ ]:
# ============================================================================
# Create Dataloaders
# ============================================================================
def create_dataloader(examples, batch_size, shuffle=False):
    """Create Grain dataloader for Tunix"""
    data_source = grain.MapDataset.source(examples)

    sampler = grain.IndexSampler(
        num_records=len(examples),
        shuffle=shuffle,
        seed=42,
        num_epochs=NUM_EPOCHS,
    )

    dataloader = grain.DataLoader(
        data_source=data_source,
        sampler=sampler,
        operations=[grain.Batch(batch_size=batch_size, drop_remainder=False)],
    )

    return dataloader

train_ds = create_dataloader(train_examples, BATCH_SIZE, shuffle=True)
eval_ds = create_dataloader(eval_examples, BATCH_SIZE, shuffle=False)

print("✓ Dataloaders created")

✓ Dataloaders created


In [ ]:
# Initialize W&B with your config
wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_RUN_NAME,
    config={
        "max_seq_length": MAX_SEQ_LENGTH,
        "batch_size": BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "effective_batch_size": BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
        "learning_rate": LEARNING_RATE,
        "num_epochs": NUM_EPOCHS,
        "warmup_steps": WARMUP_STEPS,
        "weight_decay": WEIGHT_DECAY,
        "max_grad_norm": MAX_GRAD_NORM,
        "model": "gemma3-1b-it",
        "dataset": "custom-dataset-v2",
        "training_type": "knowledge_distillation_sft",
    }
)

print(f"W&B initialized: {wandb.run.url}")

# Log to W&B
wandb.log({
    "dataset/num_train": len(train_examples),
    "dataset/num_eval": len(eval_examples),
    "dataset/train_memory_mb": total_train_mb,
    "dataset/eval_memory_mb": total_eval_mb,
})

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: hazardscarn10 (hazardscarn10-edenview) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B initialized: https://wandb.ai/hazardscarn10-edenview/tunix-sft-training-colab-v2/runs/tygh88sk


In [ ]:
# ============================================================================
# CELL 10: Setup Mesh and Training Config
# ============================================================================
# Create mesh
num_devices = len(jax.devices())
mesh = jax.make_mesh((1, num_devices), ('tp', 'fsdp'))

print(f"Mesh created: {mesh}")
print(f"Axes: {mesh.axis_names}")

# Calculate training steps
NUM_STEPS = (len(train_examples) // BATCH_SIZE // GRADIENT_ACCUMULATION_STEPS) * NUM_EPOCHS

print(f"\nTraining steps: {NUM_STEPS}")

# Checkpointing options
checkpointing_options = ocp.CheckpointManagerOptions(
    save_interval_steps=SAVE_INTERVAL_STEPS,
    max_to_keep=MAX_CHECKPOINTS_TO_KEEP,
)

# W&B metrics logging options
metrics_logging_options = MetricsLoggerOptions(
    log_dir=None,  # We're using W&B, not TensorBoard
    flush_every_n_steps=10,  # Log to W&B every 10 steps
)

# Training configuration
training_config = TrainingConfig(
    max_steps=NUM_STEPS,
    eval_every_n_steps=EVAL_INTERVAL_STEPS,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    checkpoint_root_directory=os.path.abspath(CHECKPOINT_DIR),
    checkpointing_options=checkpointing_options,
    metrics_logging_options=metrics_logging_options,
    metric_prefix="train/",
)

# Optimizer with warmup schedule
schedule = optax.warmup_cosine_decay_schedule(
    init_value=0.0,
    peak_value=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    decay_steps=NUM_STEPS - WARMUP_STEPS,
    end_value=LEARNING_RATE * 0.1,
)

optimizer = optax.chain(
    optax.clip_by_global_norm(MAX_GRAD_NORM),
    optax.scale_by_adam(
        b1=ADAM_BETA1,
        b2=ADAM_BETA2,
        eps=ADAM_EPSILON,
    ),
    optax.add_decayed_weights(WEIGHT_DECAY),
    optax.scale_by_schedule(schedule),
    optax.scale(-1.0),
)

print(f"\n✓ Training configuration created")
print(f"  Total steps: {NUM_STEPS}")
print(f"  Eval every: {EVAL_INTERVAL_STEPS} steps")
print(f"  Save every: {SAVE_INTERVAL_STEPS} steps")
print(f"  Max checkpoints kept: {MAX_CHECKPOINTS_TO_KEEP}")

Mesh created: Mesh('tp': 1, 'fsdp': 1, axis_types=(Auto, Auto))
Axes: ('tp', 'fsdp')

Training steps: 6254

✓ Training configuration created
  Total steps: 6254
  Eval every: 100 steps
  Save every: 500 steps
  Max checkpoints kept: 3


/tmp/ipython-input-2070436515.py:6: DeprecationWarning: The default axis_types will change in JAX v0.9.0 to jax.sharding.AxisType.Explicit. To maintain the old behavior, pass `axis_types=(jax.sharding.AxisType.Auto,) * len(axis_names)`. To opt-into the new behavior, pass `axis_types=(jax.sharding.AxisType.Explicit,) * len(axis_names)
  mesh = jax.make_mesh((1, num_devices), ('tp', 'fsdp'))


In [ ]:
# ============================================================================
# CELL 11: Create Trainer
# ============================================================================
# Model input function to generate attention_mask
from tunix.sft import utils

def gen_model_input_fn(batch):
    """Convert batch to model-compatible format with attention mask"""
    # batch has: input_tokens, input_mask, positions
    pad_mask = batch['input_mask'].astype(bool)  # Convert float mask to bool

    # Generate causal attention mask from pad mask
    attention_mask = utils.make_causal_attn_mask(pad_mask)

    return {
        'input_tokens': batch['input_tokens'],
        'input_mask': batch['input_mask'],
        'positions': batch['positions'],
        'attention_mask': attention_mask,
    }

# Create trainer
trainer = PeftTrainer(
    model=student_model,
    optimizer=optimizer,
    training_config=training_config,
)

# Add the model input function
trainer = trainer.with_gen_model_input_fn(gen_model_input_fn)
print("✓ Trainer ready for training")

# Check for existing checkpoints to resume
checkpoint_manager = ocp.CheckpointManager(
    CHECKPOINT_DIR,
    checkpointers=ocp.StandardCheckpointer(),
    options=checkpointing_options,
)

latest_step = checkpoint_manager.latest_step()
if latest_step is not None:
    print(f"\n⚠️  Found existing checkpoint at step {latest_step}")
    print(f"  To resume training, load the checkpoint before calling trainer.train()")
    print(f"  To start fresh, delete the {CHECKPOINT_DIR} directory")
else:
    print("\nNo existing checkpoints found. Starting fresh training.")

wandb: WARNING The anonymous setting has no effect and will be removed in a future version.


dataset/eval_memory_mb,▁
dataset/num_eval,▁
dataset/num_train,▁
dataset/train_memory_mb,▁
dataset/eval_memory_mb,28.4472
dataset/num_eval,1317
dataset/num_train,25020
dataset/train_memory_mb,540.432


✓ Trainer ready for training

No existing checkpoints found. Starting fresh training.


In [ ]:
# ============================================================================
# CELL 12: Training Loop with W&B Logging
# ============================================================================
print("="*60)
print("Starting SFT Training")
print("="*60)
print(f"Steps: {NUM_STEPS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Gradient accumulation: {GRADIENT_ACCUMULATION_STEPS}")
print(f"Effective batch: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"Training samples: {len(train_examples)}")
print(f"W&B Run: {wandb.run.url}")
print("="*60)

start_time = time.time()

# Train with mesh context
with mesh:
    trainer.train(train_ds, eval_ds)

training_time = time.time() - start_time

print(f"\n{'='*60}")
print(f"Training Complete!")
print(f"{'='*60}")
print(f"Time: {training_time/60:.2f} minutes ({training_time:.1f} seconds)")
print(f"Time per sample: {training_time/len(train_examples):.2f} sec")
print(f"Samples per second: {len(train_examples)/training_time:.2f}")

# Log final training stats to W&B
# wandb.log({
#     "training/total_time_minutes": training_time/60,
#     "training/samples_per_second": len(train_examples)/training_time,
#     "training/time_per_sample": training_time/len(train_examples),
# })

# Extrapolate to full run
full_size = 20000
est_time = (training_time / len(train_examples)) * full_size
print(f"\n{'='*60}")
print(f"Extrapolation for Kaggle TPU (20k samples):")
print(f"{'='*60}")
print(f"Estimated time: {est_time/3600:.2f} hours")
print(f"TPU budget: 9 hours")
print(f"Feasible: {'YES ✓' if est_time < 32400 else 'NO ✗ - Need to optimize'}")

# wandb.log({
#     "extrapolation/estimated_hours_20k": est_time/3600,
#     "extrapolation/feasible": est_time < 32400,
# })

Starting SFT Training
Steps: 6254
Batch size: 2
Gradient accumulation: 4
Effective batch: 8
Training samples: 25020
W&B Run: https://wandb.ai/hazardscarn10-edenview/tunix/runs/sgsnnm6r


Training:   0%|          | 0/6254 [00:00<?, ?step/s]

wandb: WARNING Tried to log to step 0 that is less than the current step 10. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: WARNING Tried to log to step 0 that is less than the current step 499. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: WARNING Tried to log to step 0 that is less than the current step 999. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: WARNING Tried to log to step 0 that is less than the current step 1499. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: WARNING Tried to log to step 0 that is less than the current step 1999. Steps must be monotonically increasing, so this data will be ignored. See https://wan

jax/checkpoint/write/blocking_gbytes_per_sec,▁
jax/core/compile/backend_compile_duration,▁
jax/core/compile/jaxpr_to_mlir_module_duration,▁
jax/core/compile/jaxpr_trace_duration,▁
jax/orbax/write/sharded_array_gb,▁
train/eval/loss,█▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/eval/perplexity,█▅▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/eval/step_time_sec,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/eval/steps_per_sec,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/train/loss,█▅▅▃█▃▃▆▅▄▇▂▅▄▄▄▇▆▄▄▁▄▃▂▃▃▁▅▃▃▅▄▃▃▂▅▆▅▄▄
+4,...



Training Complete!
Time: 226.86 minutes (13611.8 seconds)
Time per sample: 0.54 sec
Samples per second: 1.84


Error: You must call wandb.init() before wandb.log()

In [ ]:
# ============================================================================
# CELL 13: Save Final Model for Competition
# ============================================================================
print("\n" + "="*60)
print("Saving Final Model for Competition Submission")
print("="*60)



# Remove existing directory AND temp directory if they exist
import shutil
for path in [FINAL_MODEL_DIR, f"{FINAL_MODEL_DIR}.orbax-checkpoint-tmp"]:
    if os.path.exists(path):
        print(f"Removing existing directory: {path}")
        shutil.rmtree(path)
        print("✓ Removed")

# Reinitialize W&B if needed (to prevent errors during save)
if wandb.run is None:
    print("Reinitializing W&B...")
    wandb.init(
        project=WANDB_PROJECT,
        name=f"save-model-{int(time.time())}",
        resume=False
    )

with mesh:
    # Get the trained model state
    trained_state = nnx.state(trainer.model)

    # Save using StandardCheckpointer
    checkpointer = ocp.StandardCheckpointer()
    checkpointer.save(FINAL_MODEL_DIR, trained_state)
    checkpointer.wait_until_finished()

    print(f"✓ Model saved to: {FINAL_MODEL_DIR}")

# Verify what was saved
print("\nSaved files:")
for item in os.listdir(FINAL_MODEL_DIR):
    item_path = os.path.join(FINAL_MODEL_DIR, item)
    if os.path.isdir(item_path):
        print(f"  📁 {item}/")
    else:
        size_mb = os.path.getsize(item_path) / (1024*1024)
        print(f"  📄 {item} ({size_mb:.1f} MB)")

# # Save model as W&B artifact
# try:
#     artifact = wandb.Artifact(
#         name=f"gemma3-1b-distilled-model-v3",
#         type="model",
#         description="Gemma 3 1B model fine-tuned with knowledge distillation",
#         metadata={
#             "num_epochs": NUM_EPOCHS,
#             "num_steps": NUM_STEPS,
#             "learning_rate": LEARNING_RATE,
#             "training_samples": len(train_examples),
#         }
#     )
#     artifact.add_dir(FINAL_MODEL_DIR)
#     wandb.log_artifact(artifact)
#     print(f"\n✓ Model uploaded to W&B as artifact")
# except Exception as e:
#     print(f"\n⚠️  W&B upload failed: {e}")

print(f"\n✓ Model saved on Google Drive at: {FINAL_MODEL_DIR}")
print("="*60)


Saving Final Model for Competition Submission
Removing existing directory: /content/drive/MyDrive/kaggle-tunix-folder/final_model_v5
✓ Removed
Reinitializing W&B...


✓ Model saved to: /content/drive/MyDrive/kaggle-tunix-folder/final_model_v5

Saved files:
  📄 _CHECKPOINT_METADATA (0.0 MB)
  📄 _sharding (0.1 MB)
  📁 ocdbt.process_0/
  📁 array_metadatas/
  📄 _METADATA (0.1 MB)
  📁 d/
  📄 manifest.ocdbt (0.0 MB)

✓ Model saved on Google Drive at: /content/drive/MyDrive/kaggle-tunix-folder/final_model_v5


In [ ]:
# ============================================================================
# CELL 14: Load Saved Model and Test Inference
# ============================================================================
print("\n" + "="*60)
print("Loading Saved Model for Inference")
print("="*60)

# Create base model structure
print("Creating model structure...")
base_model = params.create_model_from_checkpoint(params.GEMMA3_1B_IT, model_config)

# Load the trained model we just saved
with mesh:
    # Get abstract state structure
    abs_state = jax.tree.map(
        lambda x: jax.ShapeDtypeStruct(x.shape, x.dtype) if hasattr(x, 'shape') else x,
        nnx.state(base_model),
    )

    # Load checkpoint using StandardCheckpointer
    checkpointer = ocp.StandardCheckpointer()
    loaded_state = checkpointer.restore(FINAL_MODEL_DIR, target=abs_state)

    # Update the model with trained weights
    nnx.update(base_model, loaded_state)

    print(f"✓ Model loaded from: {FINAL_MODEL_DIR}")

# Use this loaded model for inference
loaded_model = base_model

print("\n" + "="*60)
print("Testing Loaded Model Inference")
print("="*60)

sampler = sampler_lib.Sampler(
    transformer=loaded_model,
    tokenizer=gemma_tokenizer,
    cache_config=sampler_lib.CacheConfig(
        cache_size=2048,
        num_layers=model_config.num_layers,
        num_kv_heads=model_config.num_kv_heads,
        head_dim=model_config.head_dim,
    ),
)

test_prompts = [
    "What is 15 + 27?",
    "Calculate the area of a circle with radius 5.",
    "Solve for x: 2x + 5 = 13",
]

print("Generating responses...\n")

with mesh:
    for i, prompt in enumerate(test_prompts):
        full_prompt = f"<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n"

        output = sampler(
            input_strings=[full_prompt],
            max_generation_steps=256,
            temperature=0.7,
        )

        response = output.text[0]
        print(f"Test {i+1}: {prompt}")
        print(f"Response: {response[:500]}")
        print("-" * 60)

        # Log sample to W&B (if active)
        try:
            if wandb.run is not None:
                wandb.log({
                    f"inference_samples/test_{i+1}": wandb.Html(
                        f"<b>Prompt:</b> {prompt}<br><b>Response:</b> {response}"
                    )
                })
        except:
            pass

print("\n✓ Inference test complete using loaded model")


Loading Saved Model for Inference
Creating model structure...


/usr/local/lib/python3.12/dist-packages/orbax/checkpoint/_src/serialization/jax_array_handlers.py:711: UserWarning: Sharding info not provided when restoring. Populating sharding info from sharding file. Please note restoration time will be slightly increased due to reading from file. Note also that this option is unsafe when restoring on a different topology than the checkpoint was saved with.
  warnings.warn(


✓ Model loaded from: /content/drive/MyDrive/kaggle-tunix-folder/final_model_v5

Testing Loaded Model Inference
Generating responses...

Test 1: What is 15 + 27?
Response: 15 + 27 = 42.
<end_of_turn>
<start_of_turn>model
<reasoning>This is a mathematical reasoning task.

The user asks: "What is 15 + 27?" Provide answer. Provide clear concise solution. No extra. Under 500 words.</reasoning><answer>15 + 27 = 42.</answer><end_of_turn>
<start_of_turn>model
<reasoning>This is a straightforward arithmetic problem. Provide the answer.</reasoning><answer>15 + 27 = 42.</answer><end_of_turn>
<start_of_turn>model
<reasoning>The user gave a simple arithmetic problem. Provide 
------------------------------------------------------------
Test 2: Calculate the area of a circle with radius 5.
Response: <reasoning>This is a mathematical reasoning task.

We need to answer: area of a circle with radius 5. Formula: area = π * r^2 = π * 5^2 = 25π. Provide answer.

We can present as π ≈ 3.14159, maybe 25*3.1

In [ ]:
# ============================================================================
# CELL 15: Finish W&B Run
# ============================================================================
wandb.finish()
print("\n✓ W&B run finished successfully")
print(f"View your results at: {wandb.run.url}")

# ============================================================================
# CELL 16: Load Checkpoint from Specific Step (Optional)
# ============================================================================
"""
# If you want to load from a specific checkpoint step instead of final model:

import orbax.checkpoint as ocp

# Find available checkpoints
checkpoint_manager = ocp.CheckpointManager(
    CHECKPOINT_DIR,
    checkpointers=ocp.StandardCheckpointer(),
)

# Get latest step
latest_step = checkpoint_manager.latest_step()
print(f"Latest checkpoint step: {latest_step}")

# Or list all available steps
all_steps = checkpoint_manager.all_steps()
print(f"Available checkpoint steps: {all_steps}")

# Load from specific step
if latest_step is not None:
    print(f"Loading checkpoint from step {latest_step}")

    with mesh:
        from tunix.models.gemma3 import model as transformer_lib
        checkpoint_model = transformer_lib.Transformer(config=model_config)

        # Get abstract state
        abs_state = jax.tree.map(
            lambda x: jax.ShapeDtypeStruct(x.shape, x.dtype),
            nnx.state(checkpoint_model),
        )

        # Load from specific step directory
        checkpoint_path = os.path.join(CHECKPOINT_DIR, str(latest_step))
        checkpointer = ocp.StandardCheckpointer()
        loaded_state = checkpointer.restore(checkpoint_path, target=abs_state)

        # Update model
        nnx.update(checkpoint_model, loaded_state)

        print(f"✓ Checkpoint loaded from step {latest_step}")

        # Now you can use checkpoint_model for inference or continue training
else:
    print("No checkpoints found")
"""

print("\n" + "="*60)
print("Script Complete!")
print("="*60)